# Problem Statement

Unlocking Sales & Profit Insights for Sustainable Growth

Analyze XYZ Co.’s 2014–2018 sales data to uncover the key drivers of revenue and profitability across products, sales channels, and regions. Identify seasonal trends, performance gaps, and outliers, while comparing actual results against budget targets.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import pymysql 
import math
pd.set_option('display.max_columns', None)

In [4]:
sheets = pd.read_excel('Data/Regional Sales Dataset.xlsx', sheet_name = None)

# Assign dataframes to each sheet

df_sales = sheets['Sales Orders']
df_customers = sheets['Customers']
df_products = sheets['Products']
df_regions = sheets['Regions']
df_state_reg = sheets['State Regions']
df_budgets = sheets['2017 Budgets']

### Quick inspection


In [6]:
print(f"The shape of sales: {df_sales.shape}")
print(f"The shape of customers: {df_customers.shape}")
print(f"The shape of products: {df_products.shape}")
print(f"The shape of regions: {df_regions.shape}")
print(f"The shape of state regions: {df_state_reg.shape}")
print(f"The shape of budgets: {df_budgets.shape}")

The shape of sales: (64104, 12)
The shape of customers: (175, 2)
The shape of products: (30, 2)
The shape of regions: (994, 15)
The shape of state regions: (49, 3)
The shape of budgets: (30, 2)


In [9]:
df_state_reg.head(5)

,State Code,State,Region
0,AL,Alabama,South
1,AR,Arkansas,South
2,AZ,Arizona,West
3,CA,California,West
4,CO,Colorado,West


In [8]:
df_state_reg.columns = df_state_reg.iloc[0]      # set headers from row 0
df_state_reg = df_state_reg[1:].reset_index(drop=True)  # drop that row, reset index

In [14]:
df_budgets.head() 

,Product Name,2017 Budgets
0,Product 1,3016489.209
1,Product 2,3050087.565
2,Product 3,2642352.432
3,Product 4,2885560.824
4,Product 5,3925424.542


In [13]:
df_regions.head()

,id,name,county,state_code,state,type,latitude,longitude,area_code,population,households,median_income,land_area,water_area,time_zone
0,1,Auburn,Lee County,AL,Alabama,City,32.60986,-85.48078,334,62059,21767,38342,152375113,2646161,America/Chicago
1,2,Birmingham,Shelby County/Jefferson County,AL,Alabama,City,33.52744,-86.79905,205,212461,89972,31061,378353942,6591013,America/Chicago
2,3,Decatur,Limestone County/Morgan County,AL,Alabama,City,34.57332,-86.99214,256,55437,22294,41496,141006257,17594716,America/Chicago
3,4,Dothan,Dale County/Houston County/Henry County,AL,Alabama,City,31.23370,-85.40682,334,68567,25913,42426,232166237,835468,America/Chicago
4,5,Hoover,Shelby County/Jefferson County,AL,Alabama,City,33.37695,-86.80558,205,84848,32789,77146,122016784,2553332,America/Chicago


In [12]:
df_products.head()

,Index,Product Name
0,1,Product 1
1,2,Product 2
2,3,Product 3
3,4,Product 4
4,5,Product 5


In [11]:
df_customers.head()

,Customer Index,Customer Names
0,1,Geiss Company
1,2,Jaxbean Group
2,3,Ascend Ltd
3,4,Eire Corp
4,5,Blogtags Ltd


In [10]:
df_sales.head()

,OrderNumber,OrderDate,Customer Name Index,Channel,Currency Code,Warehouse Code,Delivery Region Index,Product Description Index,Order Quantity,Unit Price,Line Total,Total Unit Cost
0,SO - 000225,2014-01-01,126,Wholesale,USD,AXW291,364,27,6,2499.1,14994.6,1824.343
1,SO - 0003378,2014-01-01,96,Distributor,USD,AXW291,488,20,11,2351.7,25868.7,1269.918
2,SO - 0005126,2014-01-01,8,Wholesale,USD,AXW291,155,26,6,978.2,5869.2,684.740
3,SO - 0005614,2014-01-01,42,Export,USD,AXW291,473,7,7,2338.3,16368.1,1028.852
4,SO - 0005781,2014-01-01,73,Wholesale,USD,AXW291,256,8,8,2291.4,18331.2,1260.270


In [16]:
print(f"The info of sales: {df_sales.info()}")
print(f"The info of customers: {df_customers.info()}")
print(f"The info of products: {df_products.info()}")
print(f"The info of regions: {df_regions.info()}")
print(f"The info of state regions: {df_state_reg.info()}")
print(f"The info of budgets: {df_budgets.info()}")

<class 'pandas.DataFrame'>
RangeIndex: 64104 entries, 0 to 64103
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   OrderNumber                64104 non-null  str           
 1   OrderDate                  64104 non-null  datetime64[us]
 2   Customer Name Index        64104 non-null  int64         
 3   Channel                    64104 non-null  str           
 4   Currency Code              64104 non-null  str           
 5   Warehouse Code             64104 non-null  str           
 6   Delivery Region Index      64104 non-null  int64         
 7   Product Description Index  64104 non-null  int64         
 8   Order Quantity             64104 non-null  int64         
 9   Unit Price                 64104 non-null  float64       
 10  Line Total                 64104 non-null  float64       
 11  Total Unit Cost            64104 non-null  float64       
dtypes: datetime64[u

In [17]:
print(f"The description of sales: {df_sales.describe()}")
print(f"The description of customers: {df_customers.describe()}")
print(f"The description of products: {df_products.describe()}") 
print(f"The description of regions: {df_regions.describe()}")
print(f"The description of state regions: {df_state_reg.describe()}")
print(f"The description of budgets: {df_budgets.describe()}")

The description of sales:                         OrderDate  Customer Name Index  Delivery Region Index  \
count                       64104         64104.000000           64104.000000   
mean   2016-01-29 01:28:20.935979            87.480064             495.086609   
min           2014-01-01 00:00:00             1.000000               1.000000   
25%           2015-01-13 00:00:00            45.000000             247.000000   
50%           2016-01-27 00:00:00            87.000000             493.000000   
75%           2017-02-13 00:00:00           130.000000             742.000000   
max           2018-02-28 00:00:00           175.000000             994.000000   
std                           NaN            49.884946             285.645893   

       Product Description Index  Order Quantity    Unit Price    Line Total  \
count               64104.000000    64104.000000  64104.000000  64104.000000   
mean                   14.913141        8.441689   2284.380803  19280.682937   
min 

#### Cleaning Null value

In [18]:
print(f"The sum of null values in sales: {df_sales.isnull().sum()}")
print(f"The sum of null values in customers: {df_customers.isnull().sum()}")
print(f"The sum of null values in products: {df_products.isnull().sum()}")
print(f"The sum of null values in regions: {df_regions.isnull().sum()}")
print(f"The sum of null values in state regions: {df_state_reg.isnull().sum()}")
print(f"The sum of null values in budgets: {df_budgets.isnull().sum()}")

The sum of null values in sales: OrderNumber                  0
OrderDate                    0
Customer Name Index          0
Channel                      0
Currency Code                0
Warehouse Code               0
Delivery Region Index        0
Product Description Index    0
Order Quantity               0
Unit Price                   0
Line Total                   0
Total Unit Cost              0
dtype: int64
The sum of null values in customers: Customer Index    0
Customer Names    0
dtype: int64
The sum of null values in products: Index           0
Product Name    0
dtype: int64
The sum of null values in regions: id               0
name             0
county           0
state_code       0
state            0
type             0
latitude         0
longitude        0
area_code        0
population       0
households       0
median_income    0
land_area        0
water_area       0
time_zone        0
dtype: int64
The sum of null values in state regions: 0
State Code    0
State         0

In [20]:
dataframes = {
    "sales": df_sales,
    "customers": df_customers,
    "products": df_products,
    "regions": df_regions,
    "state regions": df_state_reg,
    "budgets": df_budgets
}

for name, df in dataframes.items():
    print(f"The duplicated rows in {name}: {df.duplicated().sum()}")

The duplicated rows in sales: 0
The duplicated rows in customers: 0
The duplicated rows in products: 0
The duplicated rows in regions: 0
The duplicated rows in state regions: 0
The duplicated rows in budgets: 0
